# Entregable 2: recuperar Colab y continuar la campaña CUDA (R3)

Esta continuación recupera los **1.552 resultados lógicos confirmados** de la campaña `e2-cuda-selection-v1`. El último intento quedó iniciado sin resultado recuperable en el Colab inspeccionado. Su desenlace, duración y pasos siguen siendo **desconocidos**.

R3 permite **una sola repetición física** del trabajo pendiente, con su configuración y semilla originales, y conserva los resultados y recibos anteriores. El nuevo resultado se registra por separado del intento perdido antes de continuar la selección. Si la repetición se interrumpe sin resultado observado, se bloquea otra repetición automática.

Selecciona **T4**. En la primera celda cambia `RUN_SEARCH=True` y `RETRY_LOST_ONCE=True` cuando quieras iniciar esta recuperación y la continuación autorizada. Después ejecuta las celdas en orden. La preparación 1–7 se conserva; la nueva sección 8 sustituye al ejecutor normal para este incidente.

El plan mantiene **24 horas iniciales** y reconstruye la ampliación existente a **36 horas efectivas** desde el diario. Esta recuperación no amplía el presupuesto. La asignación conservadora de la sesión sin cierre es contabilidad, no tiempo GPU medido. Este notebook no ejecuta el benchmark final ni fija R.


In [ ]:
from pathlib import Path
import os, re, sys, json, hashlib, subprocess, platform, shutil, math

SOURCE_COMMIT = "564ba4762f84a800b02fa4c734f5752a169f5223"
EXPECTED_SEARCH_REGISTRATION_SHA256 = "78c3c5b8ca042a347aaa80d642f3affd8b5911e8697d8747a2de915d1c3194a9"
DATA_COMMIT = "0b079b50625a936456a70695532853fc13298f31"
REPOSITORY_URL = "https://github.com/jassan-bit/DeepLearning_202630.git"

RUN_SEARCH = False
RETRY_LOST_ONCE = False
SHOW_PLAN = True
MODELS = None  # None conserva los catorce sistemas neuronales del plan.
ORIGINS = None  # None conserva los once orígenes de desarrollo admitidos.
DEVICE = "cuda:0"
SESSION_HOURS = 8
CAMPAIGN_HOURS = 24
EXTEND_CAMPAIGN_HOURS = None  # Ampliación explícita registrada al continuar; conserva el límite inicial anterior.
STORAGE_GIB = 10
CAMPAIGN_NAME = "e2-cuda-selection-v1"
GITHUB_REPOSITORY = "jassan-bit/DeepLearning_202630"
GITHUB_BRANCH_PREFIX = "campaigns"
GITHUB_PATH_PREFIX = "campaigns"

# Reutiliza el entorno y la caché del piloto; mantiene separado el código nuevo.
WORKSPACE = Path.cwd()/"entregable2_gpu_workspace"
REPO = WORKSPACE/"search-source"
VENV = WORKSPACE/"venv-py312-cu126"
DATA_CACHE = WORKSPACE/"development_cache"
LOCAL_CAMPAIGN_CACHE = WORKSPACE/"search-campaigns"
UV_VERSION = "0.9.5"
MANAGED_PYTHON_VERSION = "3.12.12"
PIP_VERSION = "25.3"
UV_WHEEL_SHA256 = "6507bbbcd788553ec4ad5a96fa19364dc0f58b023e31d79868773559a83ec181"
PIP_WHEEL_SHA256 = "9655943313a94722b7774661c21049070f6bbb0a1516bf02f7c8d5d9201514cd"

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
CUDA_ENV = dict(os.environ)
CUDA_ENV.update(CUBLAS_WORKSPACE_CONFIG=":4096:8", PYTHONNOUSERSITE="1",
                OMP_NUM_THREADS="2", MKL_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2")
CUDA_ENV.pop("PYTHONPATH", None)
CUDA_ENV.pop("GITHUB_TOKEN", None)
CUDA_ENV.pop("GH_TOKEN", None)
CUDA_ENV.update(UV_CACHE_DIR=str(WORKSPACE/"uv-cache"),
                UV_PYTHON_INSTALL_DIR=str(WORKSPACE/"managed-python"),
                UV_PYTHON_BIN_DIR=str(WORKSPACE/"managed-python-bin"), UV_NATIVE_TLS="true")
SOURCE_READY = False
PLAN_READY = False
REMOTE_READY = False
HARDWARE_READY = False
GPU_READY = False
DATA_READY = False
SYNTHETIC_READY = False
SEARCH_PLAN = None
EXECUTION_ATTEMPTED = False
LAST_SEARCH_RETURN_CODE = None
CAMPAIGN_OUTPUT = None
GITHUB_ENV = None

def run(args, *, cwd=None, capture=False, input_text=None):
    return subprocess.run([str(x) for x in args], cwd=cwd, env=CUDA_ENV, check=True,
                          text=True, input=input_text, capture_output=capture)

def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024*1024), b""):
            h.update(block)
    return h.hexdigest()


## 1. Código inmutable, sin checkout de observaciones

El modo de planificación funciona sin GPU. El checkout contiene código, pruebas y protocolos; no incluye los CSV ni carpetas de resultados. Una fuente diferente exige otro checkout y no modifica la campaña existente.


In [ ]:
SPARSE_PATHS = [
    "/scripts/", "/tests/", "/requirements-e2-training-gpu.lock.txt",
    "/book_e2/_static/search_protocol_20261109/protocol.json",
    "/experiments/dl_r2_cv_20261104/protocol.json",
    "/experiments/e2_development_pilot_20261109/source_protocol.json",
    "/experiments/e2_gpu_pilot_20261109/", "/experiments/e2_cuda_search_20261109/",
]
SOURCE_READY = False
if not re.fullmatch(r"[0-9a-f]{40}", SOURCE_COMMIT):
    raise RuntimeError("Falta el commit publicado de 40 caracteres; no usar main ni un marcador.")
if platform.system() != "Linux" or platform.machine() != "x86_64":
    raise RuntimeError("El entorno fijado corresponde a Linux x86_64.")
if not REPO.exists():
    REPO.mkdir(parents=True)
    run(["git", "init", REPO])
    run(["git", "-C", REPO, "remote", "add", "origin", REPOSITORY_URL])
    run(["git", "-C", REPO, "config", "core.sparseCheckout", "true"])
    run(["git", "-C", REPO, "config", "remote.origin.promisor", "true"])
    run(["git", "-C", REPO, "config", "remote.origin.partialclonefilter", "blob:none"])
    run(["git", "-C", REPO, "sparse-checkout", "set", "--no-cone", "--stdin"],
        input_text="\n".join(SPARSE_PATHS)+"\n")
    run(["git", "-C", REPO, "fetch", "--filter=blob:none", "--depth=1", "origin", SOURCE_COMMIT])
    run(["git", "-C", REPO, "checkout", "--detach", "FETCH_HEAD"])
if run(["git", "-C", REPO, "rev-parse", "HEAD"], capture=True).stdout.strip() != SOURCE_COMMIT:
    raise RuntimeError("El checkout existente pertenece a otro commit; usa un WORKSPACE nuevo.")
if (REPO/"results").exists():
    raise RuntimeError("Este checkout debe contener sólo código; usa una carpeta nueva sin results.")
SOURCE_READY = True
print("Fuente verificada:", SOURCE_COMMIT)


## 2. Mostrar el plan antes de la GPU o los datos

Los límites y el dominio quedan identificados antes del acceso al desarrollo. El modo `--plan-only` no crea una campaña, no abre los CSV ni importa CUDA. El límite de sesión puede dejar trabajos pendientes; el límite global no equivale a una estimación del experimento completo. Los límites científicos por etapa permanecen separados de estos techos operativos.

`ORIGINS=None` conserva los once cortes de desarrollo y `MODELS=None` los catorce sistemas neuronales. Los subconjuntos se identifican como cobertura parcial; no se presentan como todo el benchmark. Para una ampliación global posterior, conserva `CAMPAIGN_HOURS=24` y fija explícitamente `EXTEND_CAMPAIGN_HOURS`; el ejecutor registra la enmienda sin cambiar candidatos ni semillas.


In [ ]:
PLAN_READY = False
SEARCH_PLAN = None
if not SOURCE_READY:
    raise RuntimeError("Verifica primero la fuente congelada.")
if not re.fullmatch(r"[A-Za-z0-9_.-]{1,80}", CAMPAIGN_NAME) or CAMPAIGN_NAME in {".", ".."}:
    raise ValueError("Elige un nombre simple de campaña, sin rutas ni espacios.")
for value, label in [(SESSION_HOURS, "SESSION_HOURS"), (CAMPAIGN_HOURS, "CAMPAIGN_HOURS"), (STORAGE_GIB, "STORAGE_GIB")]:
    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or value <= 0:
        raise ValueError(label+" debe ser un número finito positivo.")
if SESSION_HOURS > CAMPAIGN_HOURS:
    raise ValueError("El límite de sesión no puede superar el límite global.")
if EXTEND_CAMPAIGN_HOURS is not None:
    if isinstance(EXTEND_CAMPAIGN_HOURS, bool) or not isinstance(EXTEND_CAMPAIGN_HOURS, (int, float)) or not math.isfinite(EXTEND_CAMPAIGN_HOURS) or EXTEND_CAMPAIGN_HOURS <= CAMPAIGN_HOURS:
        raise ValueError("La ampliación explícita debe superar el límite global inicial.")
CAMPAIGN_OUTPUT = LOCAL_CAMPAIGN_CACHE/CAMPAIGN_NAME
GITHUB_BRANCH = GITHUB_BRANCH_PREFIX+"/"+CAMPAIGN_NAME
GITHUB_PREFIX = GITHUB_PATH_PREFIX+"/"+CAMPAIGN_NAME
PROTOCOL_RELATIVE = "book_e2/_static/search_protocol_20261109/protocol.json"
NORMALIZED_PROTOCOL_SHA256 = "e0030c990a4afa15665aec254211b0d9e680e9ac5d92dfb8094e43e965152abe"

def verify_source_snapshot(*, check_protocol=True):
    changed = run(["git", "-C", REPO, "diff", "--name-only", "HEAD"], capture=True).stdout.splitlines()
    if any(name != PROTOCOL_RELATIVE for name in changed):
        raise RuntimeError("Hay cambios en la fuente congelada; conserva la campaña y revisa el checkout.")
    untracked = run(["git", "-C", REPO, "ls-files", "--others", "--exclude-standard", "scripts"], capture=True).stdout.splitlines()
    if any(name.endswith(".py") for name in untracked):
        raise RuntimeError("Hay módulos Python no registrados en scripts.")
    if check_protocol and sha256_file(REPO/PROTOCOL_RELATIVE) != NORMALIZED_PROTOCOL_SHA256:
        raise RuntimeError("El protocolo cambió respecto de los bytes registrados.")

verify_source_snapshot(check_protocol=False)
run([sys.executable, REPO/"scripts/prepare_e2_pilot_snapshot.py", "--root", REPO], cwd=REPO)
if sha256_file(REPO/PROTOCOL_RELATIVE) != NORMALIZED_PROTOCOL_SHA256:
    raise RuntimeError("El protocolo no coincide con los bytes registrados.")

def search_command(python, *, execute):
    command = [python, REPO/"scripts/run_e2_cuda_search.py", "--execute" if execute else "--plan-only",
               "--source-commit", SOURCE_COMMIT, "--expected-registration-sha256", EXPECTED_SEARCH_REGISTRATION_SHA256,
               "--output", CAMPAIGN_OUTPUT, "--session-hours", SESSION_HOURS,
               "--campaign-hours", CAMPAIGN_HOURS, "--storage-gib", STORAGE_GIB,
               "--device", DEVICE,
               "--github-repository", GITHUB_REPOSITORY, "--github-branch", GITHUB_BRANCH,
               "--github-prefix", GITHUB_PREFIX]
    if MODELS is not None:
        command += ["--models", *MODELS]
    if ORIGINS is not None:
        command += ["--origins", *ORIGINS]
    if execute:
        if EXTEND_CAMPAIGN_HOURS is not None:
            command += ["--extend-campaign-hours", EXTEND_CAMPAIGN_HOURS]
        command += ["--data-root", DATA_CACHE,
                    "--source-root-itransformer", REPO/"work/author_sources/iTransformer"]
    return command

if SHOW_PLAN or RUN_SEARCH:
    planned = run(search_command(sys.executable, execute=False), cwd=REPO, capture=True)
    SEARCH_PLAN = json.loads(planned.stdout)
    print(json.dumps(SEARCH_PLAN, ensure_ascii=False, indent=2, allow_nan=False))
    PLAN_READY = True
    if EXTEND_CAMPAIGN_HOURS is not None:
        print("Ampliación global solicitada al ejecutar:", EXTEND_CAMPAIGN_HOURS, "horas. Se registrará como enmienda.")
else:
    print("Activa SHOW_PLAN para revisar el dominio antes del lanzamiento.")


## 3. Configurar el guardado en GitHub

La campaña se guarda en una rama dedicada de **`jassan-bit/DeepLearning_202630`**. El disco de Colab es sólo una caché recuperable. El modo de planificación no usa credenciales ni escribe en el repositorio.

Para ejecutar desde Colab, crea un [token personal de GitHub con alcance fino](https://github.com/settings/personal-access-tokens/new) con acceso al repositorio elegido y permiso **Contents: read and write**. En el panel **Secretos** de Colab, añade `GITHUB_TOKEN` y habilita su acceso para este notebook. También puede recibirse como variable de entorno en un servidor propio. No pegues el token en una celda, un archivo de configuración o un mensaje.

La lectura del código público no necesita ese token. Sólo la sincronización de la campaña lo utiliza. Se verifica el acceso al repositorio; el ejecutor exige además confirmar en GitHub el estado de inicio antes de entrenar y el resultado antes de avanzar al siguiente trabajo. Un error de sincronización pausa la campaña. Los JSON autorizados no contienen credenciales, rutas internas ni pesos de modelos.


In [ ]:
REMOTE_READY = False
GITHUB_ENV = None

def github_execution_environment():
    import urllib.request, urllib.error
    if GITHUB_REPOSITORY != "jassan-bit/DeepLearning_202630":
        raise RuntimeError("Este notebook fija el repositorio autorizado para la campaña.")
    token = os.environ.get("GITHUB_TOKEN", "").strip()
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN").strip()
        except Exception:
            raise RuntimeError("Añade GITHUB_TOKEN en Secretos de Colab y habilita su acceso al notebook.") from None
    if not token or any(character.isspace() for character in token):
        raise RuntimeError("El secreto GITHUB_TOKEN está vacío o tiene un formato inválido.")
    class NoRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, request, response, code, message, headers, new_url):
            return None
    request = urllib.request.Request("https://api.github.com/repos/"+GITHUB_REPOSITORY,
        headers={"Authorization":"Bearer "+token, "Accept":"application/vnd.github+json",
                 "X-GitHub-Api-Version":"2022-11-28"}, method="GET")
    try:
        with urllib.request.build_opener(NoRedirect()).open(request, timeout=30) as response:
            raw = response.read(65537)
            if len(raw) > 65536:
                raise ValueError("Unexpected response size")
            payload = json.loads(raw)
    except urllib.error.HTTPError as error:
        raise RuntimeError("GitHub rechazó la comprobación de acceso (HTTP "+str(error.code)+"). Revisa el secreto y su repositorio autorizado.") from None
    except Exception:
        raise RuntimeError("No se pudo comprobar el acceso a GitHub; no se habilitó el entrenamiento.") from None
    if not isinstance(payload, dict) or payload.get("full_name", "").lower() != GITHUB_REPOSITORY.lower():
        raise RuntimeError("GitHub no confirmó el repositorio esperado.")
    environment = dict(CUDA_ENV)
    environment["GITHUB_TOKEN"] = token
    return environment

if RUN_SEARCH:
    if not PLAN_READY:
        raise RuntimeError("Revisa primero el plan sin entrenamiento.")
    GITHUB_ENV = github_execution_environment()
    REMOTE_READY = True
    print("Acceso al repositorio confirmado. El ejecutor comprobará el guardado remoto antes de cada ajuste.")
else:
    print("Planificación: no se leyeron secretos ni se escribió en GitHub.")


## 4. Admitir una GPU real antes de instalar CUDA

Activa GPU en Colab antes de `RUN_SEARCH=True`. Una biblioteca CUDA instalada no basta. Esta sección comprueba Linux x86_64 y una tarjeta NVIDIA; el entorno aislado comprobará después una operación CUDA real. No hay sustitución automática por CPU.


In [ ]:
HARDWARE_READY = False
GPU_READY = False
DATA_READY = False
SYNTHETIC_READY = False
if RUN_SEARCH:
    if not REMOTE_READY or not PLAN_READY:
        raise RuntimeError("Verifica el plan y el guardado en GitHub antes de instalar CUDA.")
    if not re.fullmatch(r"[0-9a-f]{40}", SOURCE_COMMIT):
        raise RuntimeError("SOURCE_COMMIT debe contener el SHA GPU publicado de 40 caracteres; no usar main.")
    if platform.system() != "Linux" or platform.machine() != "x86_64":
        raise RuntimeError("El lock CUDA corresponde a Linux x86_64.")
    if shutil.which("nvidia-smi") is None:
        raise RuntimeError("No hay GPU NVIDIA visible. Activa GPU en el proveedor antes de continuar.")
    probe = run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture=True)
    if not probe.stdout.strip():
        raise RuntimeError("nvidia-smi no informó ninguna GPU.")
    print(probe.stdout.strip())
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    HARDWARE_READY = True
    print("Python del kernel:", ".".join(map(str, sys.version_info[:3])))
else:
    print("Planificación: no se consultó ni inicializó una GPU.")


## 5. Reutilizar el Python aislado y comprobar CUDA

Se reutiliza el entorno del piloto si sus versiones coinciden. El bootstrap fija uv 0.9.5, CPython 3.12.12 y pip 25.3, comprueba los wheels por SHA256 e instala el lock PyTorch 2.8.0 + CUDA 12.6. No modifica paquetes del kernel anfitrión. La instalación y las pruebas previas no forman parte del cronómetro del ejecutor; tampoco definen el tiempo cobrado o la cuota de GPU del proveedor.


In [ ]:
GPU_READY = False
DATA_READY = False
SYNTHETIC_READY = False
if RUN_SEARCH:
    if not HARDWARE_READY:
        raise RuntimeError("Comprueba primero Linux y la GPU NVIDIA en la sección 1.")
    
    def bootstrap_python312():
        bootstrap = WORKSPACE/"bootstrap"
        bootstrap.mkdir(parents=True, exist_ok=True)
        uv_home = bootstrap/("uv-"+UV_VERSION)
        uv_binary = uv_home/"bin/uv"
        uv_requirements = bootstrap/"uv.requirements.txt"
        uv_requirements.write_text("uv=="+UV_VERSION+" --hash=sha256:"+UV_WHEEL_SHA256+"\n", encoding="utf-8")
        if not uv_binary.is_file():
            if uv_home.exists():
                raise RuntimeError("El bootstrap uv está incompleto; elige un WORKSPACE nuevo.")
            run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-cache-dir",
                 "--no-deps", "--only-binary=:all:", "--require-hashes", "--index-url", "https://pypi.org/simple",
                 "--target", uv_home, "-r", uv_requirements])
        uv_version = run([uv_binary, "--version"], capture=True).stdout.strip().split()
        if uv_version[:2] != ["uv", UV_VERSION]:
            raise RuntimeError("La versión local de uv no coincide con la fijada.")
        managed_directory = WORKSPACE/"managed-python"
        run([uv_binary, "python", "install", MANAGED_PYTHON_VERSION,
             "--install-dir", managed_directory, "--no-bin", "--no-config"])
        managed_python = Path(run([uv_binary, "python", "find", MANAGED_PYTHON_VERSION,
                                  "--managed-python", "--no-python-downloads", "--no-project", "--no-config"],
                                 capture=True).stdout.strip())
        if managed_directory.resolve() not in managed_python.resolve().parents:
            raise RuntimeError("uv seleccionó un Python fuera del directorio gestionado.")
        if not (VENV/"bin/python").is_file():
            run([uv_binary, "venv", "--python", managed_python, "--no-python-downloads", "--no-project",
                 "--no-config", VENV])
        python = VENV/"bin/python"
        info = json.loads(run([python, "-c", "import sys,json; print(json.dumps({'version':list(sys.version_info[:3])}))"],
                              capture=True).stdout)
        if info["version"] != [3, 12, 12]:
            raise RuntimeError("El venv no usa CPython 3.12.12; elige una carpeta nueva.")
        pip_requirements = bootstrap/"pip.requirements.txt"
        pip_requirements.write_text("pip=="+PIP_VERSION+" --hash=sha256:"+PIP_WHEEL_SHA256+"\n", encoding="utf-8")
        run([uv_binary, "pip", "install", "--python", python, "--no-deps", "--only-binary=:all:",
             "--require-hashes", "--index-url", "https://pypi.org/simple", "--no-config", "-r", pip_requirements])
        run([python, "-c", "import pip; assert pip.__version__=='25.3'"])
        print("Python del kernel:", ".".join(map(str, sys.version_info[:3])))
        print("Python aislado del buscador:", ".".join(map(str, info["version"])), "con pip", PIP_VERSION)
        return python
    
    GPU_PYTHON = bootstrap_python312()
    run([GPU_PYTHON, "-m", "pip", "install", "-r", REPO/"requirements-e2-training-gpu.lock.txt"])
    run([GPU_PYTHON, "-m", "pip", "check"])
    run([GPU_PYTHON, "-c", "import sys,torch; assert sys.version_info[:3]==(3,12,12); assert torch.__version__=='2.8.0+cu126'; assert torch.version.cuda=='12.6'; print('Python/Torch/CUDA: versiones verificadas')"])
    # Sólo la prueba CUDA real habilita las siguientes celdas; no se abren datos aquí.
    run([GPU_PYTHON, REPO/"scripts/run_e2_gpu_pilot.py", "--check-environment", "--device", DEVICE], cwd=REPO)
    GPU_READY = True
else:
    print("Planificación: no se instalaron paquetes ni se ejecutaron kernels CUDA.")


## 6. Fuentes y pruebas artificiales antes de los CSV

Se verifican las ocho fuentes oficiales de iTransformer. Una caché válida puede reutilizarse; los archivos o recibos diferentes se conservan y bloquean el lanzamiento. Las pruebas artificiales de CUDA y del adaptador se ejecutan antes de descargar observaciones. Sus ajustes sintéticos no son resultados HPO/NAS sobre mercado.


In [ ]:
SYNTHETIC_READY = False
DATA_READY = False
def ensure_itransformer_sources():
    import ast
    source = REPO/"scripts/fetch_e2_itransformer_sources.py"
    output = REPO/"work/author_sources/iTransformer"
    def reject_symlinks(path):
        if any(part.is_symlink() for part in (path.absolute(), *path.absolute().parents)):
            raise RuntimeError("El árbol iTransformer no admite enlaces simbólicos.")
    reject_symlinks(source)
    reject_symlinks(output)
    frozen = run(["git", "-C", REPO, "show", SOURCE_COMMIT+":scripts/fetch_e2_itransformer_sources.py"], capture=True).stdout
    if source.read_bytes() != frozen.encode():
        raise RuntimeError("El descargador iTransformer difiere de la fuente congelada.")
    constants = {node.targets[0].id: ast.literal_eval(node.value)
                 for node in ast.parse(frozen).body if isinstance(node, ast.Assign)
                 and len(node.targets) == 1 and isinstance(node.targets[0], ast.Name)
                 and node.targets[0].id == "MANIFEST"}
    manifest, maximum = constants["MANIFEST"], 1024*1024
    if not output.exists():
        run([GPU_PYTHON, source, "--output", "work/author_sources/iTransformer"], cwd=REPO)
    if not output.is_dir():
        raise RuntimeError("El destino iTransformer no es un directorio válido.")
    for path in output.rglob("*"):
        reject_symlinks(path)
        relative = path.relative_to(output).as_posix()
        if path.is_file() and path.suffix in {".py", ".pyc", ".pyo", ".pth", ".so"} and relative not in manifest["files"]:
            if "__pycache__" not in path.relative_to(output).parts or path.suffix != ".pyc":
                raise RuntimeError("Hay código no registrado en el árbol iTransformer.")
    for relative, expected in manifest["files"].items():
        path = output/relative
        reject_symlinks(path)
        if not path.is_file() or path.stat().st_size > maximum or sha256_file(path) != expected:
            raise RuntimeError("Fuente iTransformer ausente o con hash distinto: "+relative)
    receipt = output/"source_receipt.json"
    reject_symlinks(receipt)
    expected_receipt = {"status":"complete", "repository":manifest["repository"], "commit":manifest["commit"],
                        "verified_files":manifest["files"], "weights_downloaded":False, "market_data_read":False}
    if not receipt.is_file() or receipt.stat().st_size > maximum or json.loads(receipt.read_text()) != expected_receipt:
        raise RuntimeError("El recibo iTransformer no coincide con los ocho archivos registrados.")
    print("iTransformer: ocho fuentes verificadas; caché reutilizable.")

if RUN_SEARCH:
    if not GPU_READY or not REMOTE_READY:
        raise RuntimeError("Comprueba CUDA y el acceso remoto antes de los tests.")
    ensure_itransformer_sources()
    smoke_env = dict(CUDA_ENV)
    smoke_env["E2_ITRANSFORMER_SOURCE_ROOT"] = str(REPO/"work/author_sources/iTransformer")
    for pattern in ["test_e2_cuda_models.py", "test_e2_cuda_training.py", "test_e2_cuda_bindings.py", "test_e2_search_session.py"]:
        subprocess.run([str(GPU_PYTHON), "-m", "unittest", "discover", "-s", "tests", "-p", pattern, "-v"],
                       cwd=REPO, env=smoke_env, check=True)
    SYNTHETIC_READY = True
    print("Comprobaciones artificiales completadas antes del acceso al desarrollo.")
else:
    print("Planificación: no se descargaron fuentes de autor ni se ajustaron modelos sintéticos.")


## 7. Obtener únicamente el desarrollo registrado

Con la ejecución desactivada no se descargan CSV. Al activarla se admiten sólo los derivados de 2023–2024 y su protocolo fijados por hash. El helper puede reutilizar la caché existente y rechaza bytes distintos. La selección recorre los once orígenes declarados en el plan, o el subconjunto explícito de `ORIGINS`. El origen 2025-01-01 es un marcador de calendario que utiliza sólo desarrollo disponible hasta 2024; sus valores finales no se abren.


In [ ]:
DATA_READY = False
DATA_FILES = {
    "results/eda_development/daily_market_development.csv": "bf6bb5160af924aef6a15c3684aeae932df3017e8639888bb7e91ce787900171",
    "results/eda_development/daily_volatility_development.csv": "20b96374a659eb87dadfa68aa3d6b6b5fb6bea5b10eebe6f2c186ad69861844a",
}
if RUN_SEARCH:
    if not GPU_READY or not PLAN_READY or not REMOTE_READY or not SYNTHETIC_READY:
        raise RuntimeError("Verifica primero el plan, CUDA, las pruebas artificiales y el guardado en GitHub.")
    run([GPU_PYTHON, REPO/"scripts/fetch_e2_development_sources.py", "--output", DATA_CACHE], cwd=REPO)
    for relative, checksum in DATA_FILES.items():
        if sha256_file(DATA_CACHE/relative) != checksum:
            raise RuntimeError("El derivado no coincide con el SHA256 registrado.")
    DATA_READY = True
    print("Desarrollo verificado; no se descargaron valores finales.")
else:
    print("Planificación: no se descargaron los CSV.")


## 8. Recuperar el intento pendiente y continuar la selección

El controlador verifica la fuente A3, los hashes, la ampliación de presupuesto y el estado remoto antes de entrenar. Restaura una caché ausente desde la campaña existente; no inicia otra campaña. Registra recibos físicos de creación única para el trabajo autorizado.

Mantén esta celda en ejecución. Su salida y el PID mostrado permiten revisar el proceso. Al terminar, revisa el estado: una pausa de presupuesto o de sincronización no significa que el benchmark esté completo. No vuelvas a ejecutar esta celda mientras siga activa.


In [ ]:
# Recuperación publicada y fijada; ejecución sólo con las dos opciones activadas.
RECOVERY_COMMIT = "a7807a2e31030516d370cd47acd7f06ab7aef076"
EXPECTED_RECOVERY_AUTHORIZATION_SHA256 = "005090e95dd45c3358b3f07bb31716c110740f7502bcad9baa520e447fab9ca9"
EXPECTED_RECOVERY_LAUNCHER_SHA256 = "ab8a6dd78e06970a7e31e7ca5d10ae2fa26713a60b44da904dccf1caf10ea07a"

if not (RUN_SEARCH and RETRY_LOST_ONCE):
    print("Modo preparación: activa RUN_SEARCH y RETRY_LOST_ONCE en la primera celda para recuperar y continuar.")
else:
    import types
    import urllib.request
    url = ("https://raw.githubusercontent.com/jassan-bit/DeepLearning_202630/"
           + RECOVERY_COMMIT + "/scripts/colab_recovery_v3_launcher.py")
    request = urllib.request.Request(url, headers={"User-Agent": "E2-Colab-Recovery-R3/1.0"})
    with urllib.request.urlopen(request, timeout=30) as response:
        launcher_bytes = response.read(256 * 1024 + 1)
    if len(launcher_bytes) > 256 * 1024 or hashlib.sha256(launcher_bytes).hexdigest() != EXPECTED_RECOVERY_LAUNCHER_SHA256:
        raise RuntimeError("El lanzador R3 no coincide con los bytes publicados; no se inició la recuperación.")
    recovery_module = types.ModuleType("e2_recovery_v3_launcher")
    recovery_module.__file__ = url
    exec(compile(launcher_bytes, url, "exec"), recovery_module.__dict__)
    RECOVERY_V3_RESULT = recovery_module.run_launcher(globals())


## 9. Revisar el resultado de esta invocación

Este informe corresponde a la invocación R3 de este kernel. Un `status.json` de una sesión anterior puede quedar desactualizado y no demuestra actividad actual. Los tiempos y pasos del intento perdido continúan desconocidos aunque la repetición produzca un resultado.


In [ ]:
history = globals().get("E2_RECOVERY_V3_RUN_HISTORY", [])
if history:
    print(json.dumps(history[-1], ensure_ascii=False, indent=2, allow_nan=False))
else:
    print("Este kernel aún no registra una ejecución R3; revisa la salida de la sección 8.")
